### Adding Macroeconomic Variables to a Simple Model

In [101]:
import os 
cwd = os.getcwd()
cwd

os.chdir("/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline")

new_cwd = os.getcwd() 

new_cwd


'/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Git Files/msc-project-source-code-files-24-25-jmthomas565/Code/Project Code Pipeline'

In [102]:
%run 00_inititial_transformations.ipynb

['2017-07-01', '2017-07-02', '2017-07-03', '2017-07-04', '2017-07-05'] ... ['2022-09-27', '2022-09-28', '2022-09-29', '2022-09-30', '2022-10-01']


In [103]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler


In [104]:
def adstock(x, theta):
    return tsa.filters.recursive_filter(x, theta)

#locking in final AdStock scores, on cols that need it
X_train["branded_search_spend_adstock"] = adstock(X_train["branded_search_spend"], [0.9])
X_train["nonbranded_search_spend_adstock"] = adstock(X_train["nonbranded_search_spend"], [0.9])
X_train["facebook_spend_adstock"] = adstock(X_train["facebook_spend"], [0.9])

#Creating the lags for macro variables
X_train['gdp_lag_1'] = X_train['gdp'].shift(-1)
X_train['base_rate_lag_1'] = X_train['base_rate'].shift(-1)
X_train['UNRATENSA_daily_lag_1'] = X_train['UNRATENSA_daily'].shift(-1)
X_train['sales_lag_1'] = y_train['sales'].shift(-1)

#Loading only the columns that are required into the df. 
#Note that the missing offline channels are left out, because the baseline and adstock models found them not required.
X_train_final = X_train[[
    "branded_search_spend_adstock",
    'nonbranded_search_spend_adstock',
    'facebook_spend_adstock',
    'print_spend',
    'gdp', 
    'base_rate', 
    'UNRATENSA_daily',
    'sales_lag_1',
    'gdp_lag_1', 
    'base_rate_lag_1', 
    'UNRATENSA_daily_lag_1']]

#Setting the dfs from the 2nd row, because the lags have created null rows. 
X_train_final = X_train_final.iloc[:-1]

y_train = y_train.iloc[:-1]

In [105]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

X_train_final = pd.DataFrame(
    scaler.fit_transform(X_train_final),
    columns=X_train_final.columns, 
    index=X_train_final.index   
)


In [106]:
#Cross-validation hold-out
split_point = -26
y_train_cv = y_train.iloc[:split_point]
y_val_cv = y_train.iloc[split_point:]

X_train_subset = X_train_final.iloc[:split_point]
X_val_subset = X_train_final.iloc[split_point:]

In [107]:
#Calculating adj_R^2

def adj_r2(y_true, y_pred, p):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    #Making sure the denominator of adj R^2 > 0
    if n - p - 1 <= 0:
        return np.nan
    calculation = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return calculation

#Instantiating and fitting the model
ridge = Ridge(alpha=1, positive = True)
ridge.fit(X_train_subset, y_train_cv)

#Predicting
y_pred = ridge.predict(X_val_subset)

#Printing out adj_R^2 value
adj_r2 = adj_r2(y_val_cv, y_pred, p=X_train_subset.shape[1])
print(f"Full model: Adjusted R² = {adj_r2:.4f}")


Full model: Adjusted R² = 0.5814
